# Science Tutor LoRA — environment setup

This notebook installs a pinned project revision and validates the planned smoke
experiment. It does **not** download model weights or start training. A CPU
runtime is enough for this milestone; select a GPU when the training notebook
is introduced.

Run the cells in order in Google Colab. Keep notebook outputs out of Git.


In [ ]:
import platform
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("This project currently supports Python 3.11–3.13.")

print(f"Python: {platform.python_version()} | {platform.system()} {platform.machine()}")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=True)
else:
    print("No NVIDIA GPU detected. CPU is sufficient for setup and validation.")

## Install an immutable checkout

The commit below contains the tested package and lockfile. Each run creates a
fresh temporary checkout. Record the printed commit and keep the same value to
repeat this setup. A future experiment may deliberately select a newer commit.


In [ ]:
PROJECT_URL = "https://github.com/ludthor/science-tutor-lora.git"
PROJECT_REF = "c668f13f806c4a4957400ef29f821093381b6713"
PROJECT_ROOT = Path(tempfile.mkdtemp(prefix="science-tutor-")) / "repository"

subprocess.run(["git", "clone", "--no-checkout", PROJECT_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(["git", "checkout", "--detach", PROJECT_REF], cwd=PROJECT_ROOT, check=True)
resolved_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if resolved_commit != PROJECT_REF:
    raise RuntimeError("The checkout does not match the pinned project revision.")
print(f"Project directory: {PROJECT_ROOT}")
print(f"Project commit: {resolved_commit}")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.5.16"], check=True)
UV = [sys.executable, "-m", "uv"]
subprocess.run([*UV, "sync", "--locked", "--python", sys.executable], cwd=PROJECT_ROOT, check=True)

In [ ]:
for command in [
    ["science-tutor", "doctor"],
    ["science-tutor", "validate", "configs/smoke.toml"],
    ["pytest", "-q"],
]:
    subprocess.run(
        [*UV, "run", "--python", sys.executable, "--no-sync", *command],
        cwd=PROJECT_ROOT,
        check=True,
    )

## What comes next

The next milestone downloads and audits ScienceQA at its pinned revision,
records eligible examples, and verifies split isolation. Baseline evaluation
and GPU training follow that audit. No accuracy, training cost, or memory
measurements are available yet.

Temporary files disappear with the runtime. Training will add checkpoint export
and resume instructions before longer sessions are attempted.
